In [1]:
# %% [markdown]
# # Ray Tune 最终 Epoch 结算分析函数 (含 Temp 温度/EFIN/S4 全兼容版)
# 逻辑：传入 base_path，动态抓取 head_hidden_dims、alpha 以及各版本特异性 temp 温度指标。

# %%
import os
import json
import pandas as pd

# %%
def analyze_ray_results(base_path):
    """
    遍历指定的 Ray 结果路径，按模型实际超参分类对比最后 Epoch 的结算指标。
    🌟 完美补齐：自动提取 ours_s6_temp 等各种冲突温度指标，拒绝张冠李戴。
    """
    print(f"===========================================================")
    print(f" 开始解析路径: {base_path}")
    print(f"===========================================================")
    
    all_trials_data = []

    # 遍历目录
    for root, dirs, files in os.walk(base_path):
        if "params.json" in files and "result.json" in files:
            params_path = os.path.join(root, "params.json")
            result_path = os.path.join(root, "result.json")
            
            # 1. 解析参数
            try:
                with open(params_path, 'r', encoding='utf-8') as f:
                    params = json.load(f)
            except Exception:
                continue
                
            # 2. 定位最后一轮的结算指标
            last_epoch_data = None
            try:
                with open(result_path, 'r', encoding='utf-8') as f:
                    lines = [line.strip() for line in f if line.strip()]
                    if lines:
                        # 直接获取最后一行（最后 Epoch）
                        last_epoch_data = json.loads(lines[-1])
            except Exception:
                continue
                
            if last_epoch_data is not None:
                # 转换 list 为 str 方便后续做分类聚合
                head_dims = params.get("head_hidden_dims")
                head_dims_str = str(head_dims) if head_dims is not None else "None"
                
                # 提取最终结算数据 (引入全部新模型变量与核心 temp 变量兜底)
                trial_info = {
                    "trial_id": last_epoch_data.get("trial_id", os.getpid()),
                    "model": params.get("model", "None"),
                    "efin_embed_dim": params.get("efin_embed_dim", "None"), 
                    "head_hidden_dims": head_dims_str,                     
                    "alpha_wool": params.get("conflict_alpha_wool", "None"),
                    "alpha_gold": params.get("conflict_alpha_gold", "None"),
                    "alpha_walkin": params.get("conflict_alpha_walkin", "None"),
                    # 🌟 绝杀锁定：提取大盘消融矩阵中的各类温度参数 (支持 s6_temp 等)
                    "temp": params.get("ours_s6_temp", params.get("truncation_temp", params.get("align_temp", "None"))),
                    "weight_decay": params.get("weight_decay", "None"),
                    "final_epoch": last_epoch_data.get("epoch"),
                    "final_local_best_test_Y_AUUC": last_epoch_data.get("local_best_test_Target_Y_AUUC"),
                    "final_local_best_test_C_AUUC": last_epoch_data.get("local_best_test_Target_C_AUUC"),
                    "final_loss": last_epoch_data.get("loss")
                }
                all_trials_data.append(trial_info)

    # 3. 统计并打印结果
    df = pd.DataFrame(all_trials_data)

    if df.empty:
        print("❌ 没有找到任何有效的实验数据，请检查传入的路径是否正确。\n")
        return None
        
    # 🌟 关键补齐：将核心分类参数加入 temp
    core_params = ["model","head_hidden_dims", "alpha_wool", "alpha_gold", "alpha_walkin", "temp", "weight_decay"]
    for col in core_params:
        if col in df.columns:
            df[col] = df[col].fillna("None").astype(str)
        else:
            df[col] = "None"

    print(f"📊 成功加载 {len(df)} 个实验的最终结算数据。\n")
    
    # -------------------------------------------------------------
    # 维度 1：按参数组合展现各自最后 Epoch 的最高表现
    # -------------------------------------------------------------
    print("="*75)
    print(" 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC ")
    print("="*75)
    
    idx = df.groupby(core_params)["final_local_best_test_Y_AUUC"].idxmax()
    df_best_per_group = df.loc[idx].sort_values(by="final_local_best_test_Y_AUUC", ascending=False)
    
    for _, row in df_best_per_group.iterrows():
        print(f"\n[配置组合] Model: {row['model']}  | Head: {row['head_hidden_dims']}")
        print(f"            alpha_w/g/wk: {row['alpha_wool']}/{row['alpha_gold']}/{row['alpha_walkin']} | Temp: {row['temp']} | wd: {row['weight_decay']}")
        print(f"  -> 最终结算 Epoch: {row['final_epoch']} (Loss: {row['final_loss']:.4f})")
        print(f"  -> 最终 local_best_test_Y_AUUC: {row['final_local_best_test_Y_AUUC']:.6f}")
        print(f"  -> 最终 local_best_test_C_AUUC: {row['final_local_best_test_C_AUUC']:.6f}")
        print(f"  -> Trial ID: {row['trial_id']}")

    # -------------------------------------------------------------
    # 维度 2：大宽表横向对比
    # -------------------------------------------------------------
    print("\n" + "="*75)
    print(" 📋 全局最终战报总览表（按最终 local_best_test_Y_AUUC 从高到低排序） ")
    print("="*75)
    
    report_cols = core_params + ["final_epoch", "final_local_best_test_Y_AUUC", "final_local_best_test_C_AUUC"]
    print(df.sort_values(by="final_local_best_test_Y_AUUC", ascending=False)[report_cols].to_string(index=False))
    print("\n")
    
    return df

In [2]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s4_conflict_0713_h32_search_wd_alpha"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s4_conflict_0713_h32_search_wd_alpha


📊 成功加载 124 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 5.0/5.0/5.0 | Temp: None | wd: 0.0001
  -> 最终结算 Epoch: 13 (Loss: 0.0435)
  -> 最终 local_best_test_Y_AUUC: 0.910228
  -> 最终 local_best_test_C_AUUC: 0.828821
  -> Trial ID: f3380_00013

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 1.0/1.0/1.0 | Temp: None | wd: 0.01
  -> 最终结算 Epoch: 14 (Loss: 0.0253)
  -> 最终 local_best_test_Y_AUUC: 0.909630
  -> 最终 local_best_test_C_AUUC: 0.836437
  -> Trial ID: f3380_00027

[配置组合] Model: TARNET  | Head: [64, 32]
            alpha_w/g/wk: 0.1/0.1/0.1 | Temp: None | wd: 1e-05
  -> 最终结算 Epoch: 18 (Loss: 0.0132)
  -> 最终 local_best_test_Y_AUUC: 0.909402
  -> 最终 local_best_test_C_AUUC: 0.819737
  -> Trial ID: 3ca87_00011

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 5.0/5.0/5.0 | Temp: None | wd: 1e-05
  -> 最终结算 Epoch: 13 (Loss: 0.0436)
  -> 最终 local_best_test_Y_AUUC: 0.908957
  -> 最终 local_be

In [3]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0713_h32_search_wd_alpha_temp"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0713_h32_search_wd_alpha_temp


📊 成功加载 176 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 0.1/0.1/0.1 | Temp: 1 | wd: 0.001
  -> 最终结算 Epoch: 15 (Loss: 0.0153)
  -> 最终 local_best_test_Y_AUUC: 0.910478
  -> 最终 local_best_test_C_AUUC: 0.827250
  -> Trial ID: 94c1a_00002

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 5.0/5.0/5.0 | Temp: 1 | wd: 0.001
  -> 最终结算 Epoch: 13 (Loss: 0.0471)
  -> 最终 local_best_test_Y_AUUC: 0.908882
  -> 最终 local_best_test_C_AUUC: 0.823290
  -> Trial ID: 94c1a_00001

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 0.1/0.1/0.1 | Temp: 1 | wd: 1e-05
  -> 最终结算 Epoch: 15 (Loss: 0.0135)
  -> 最终 local_best_test_Y_AUUC: 0.908665
  -> 最终 local_best_test_C_AUUC: 0.830666
  -> Trial ID: 03cdf_00002

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 5.0/5.0/5.0 | Temp: 1 | wd: 1e-05
  -> 最终结算 Epoch: 13 (Loss: 0.0440)
  -> 最终 local_best_test_Y_AUUC: 0.908496
  -> 最终 local_best_test_C_AUUC: 

In [4]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_0713_h32_search_search_alpha"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_0713_h32_search_search_alpha
📊 成功加载 127 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET_Baseline_PureV10  | Head: None
            alpha_w/g/wk: 0.0/0.0/0.0 | Temp: None | wd: 0.001
  -> 最终结算 Epoch: 15 (Loss: 0.0143)
  -> 最终 local_best_test_Y_AUUC: 0.910514
  -> 最终 local_best_test_C_AUUC: 0.830121
  -> Trial ID: 37e85_00017

[配置组合] Model: TARNET_Baseline_PureV10  | Head: None
            alpha_w/g/wk: 0.01/0.01/0.01 | Temp: None | wd: 0.001
  -> 最终结算 Epoch: 15 (Loss: 0.0144)
  -> 最终 local_best_test_Y_AUUC: 0.910509
  -> 最终 local_best_test_C_AUUC: 0.829840
  -> Trial ID: 37e85_00018

[配置组合] Model: TARNET_Baseline_PureV10  | Head: None
            alpha_w/g/wk: 0.1/0.1/0.1 | Temp: None | wd: 0.001
  -> 最终结算 Epoch: 15 (Loss: 0.0154)
  -> 最终 local_best_test_Y_AUUC: 0.910469
  -> 最终 local_best_test_C_AUUC: 0.834293
  -> Trial ID: 37e85_00023

[配置组合] Model: TAR